## Imports

In [1]:
from ultralytics import YOLO
import pandas as pd

import cv2
import numpy as np
from matplotlib import pyplot as plt

from PIL import Image

In [2]:
import onnxruntime as ort

opts = ort.SessionOptions()
opts.intra_op_num_threads = 1
opts.inter_op_num_threads = 1

## Loading the model

In [ ]:
onnx_model = YOLO("/home/mrajaraman/master-thesis-dragonfly/yolo-model/runs/segment/train60/weights/best.onnx")

In [ ]:
test_images = pd.read_csv("/home/mrajaraman/master-thesis-dragonfly/test_batch.csv")

In [ ]:
test_image = test_images['Path'][0]

In [ ]:
results = onnx_model(test_image, save=True)

In [ ]:
r = results[0]
masks = r.masks  # segmentation masks
test_image = cv2.imread(test_image)
img = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)


if masks is not None:
    for mask in masks.data:
        m = mask.cpu().numpy()

        # Resize mask to match the image shape
        m = cv2.resize(m, (test_image.shape[1], test_image.shape[0]))  # (width, height)
        m = (m * 255).astype(np.uint8)

        color = np.random.randint(0, 255, (3,), dtype=np.uint8)
        test_image[m > 0] = test_image[m > 0] * 0.5 + color * 0.5

    cv2.imshow("Masks Overlay", test_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("⚠️ No masks found — ONNX export might not include segmentation outputs.")

In [ ]:
for r in results:
    print(r.masks)   # segmentation masks (if available)
    print(r.boxes)   # bounding boxes
    print(r.names)   # class names


## Loading image for extraction

In [ ]:
test_image = cv2.imread(test_image)
img = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
plt.imshow(img)

In [ ]:
print(results[0])

In [ ]:
simple_mask = results[0].masks.data[2].cpu().numpy()  # [H,W] binary mask

In [ ]:
np.unique(simple_mask)

In [ ]:
print("mask shape:", simple_mask.shape)
print("img shape:", img.shape)

In [ ]:
mask_resized = cv2.resize(simple_mask, (img.shape[1], img.shape[0]))  # (width, height)
result = cv2.bitwise_and(img, img, mask=mask_resized.astype('uint8'))

In [ ]:
# mask_resized = cv2.resize(simple_mask, (img.shape[1], img.shape[0]))

# # 2️⃣ Ensure mask is 8-bit unsigned integer
# mask_uint8 = simple_mask.astype(np.uint8)

# 3️⃣ If mask is not binary (e.g., values 0–1 or 0–255), threshold it
_, mask_binary = cv2.threshold(simple_mask, 127, 255, cv2.THRESH_BINARY)

# 4️⃣ Apply bitwise_and
res = cv2.bitwise_and(img, img, mask=mask_binary)


In [ ]:
print(results)

In [ ]:
# print(results[1][0].masks)

In [ ]:
hsv_img = cv.cvtColor(img, cv.COLOR_BGR2HSV)

In [ ]:
head_masks = results[0].masks.data[0].cpu().numpy() 

In [ ]:
head_masks_resized = cv.resize(
    head_masks,
    (img.shape[1], img.shape[0]),  # (width, height)
    interpolation=cv.INTER_NEAREST  # important to keep mask binary
)

In [ ]:
print("mask shape:", head_masks.shape)
print("img shape:", hsv_img.shape)

In [ ]:
# masks = Image.fromarray(head_masks,0)

In [ ]:
masks = 255 - masks

In [ ]:
mask_inv_3ch = cv.merge([head_masks_resized]*3)

# Now multiply
final_image = mask_inv_3ch * hsv_img

In [ ]:
plt.imshow(final_image)

In [ ]:
mask_uint8 = (head_masks.astype(np.uint8)) * 255

In [ ]:
mask_inv = 255 - head_masks

In [ ]:
# mask_uint8 = head_masks_resized.astype(np.uint8)

# 3️⃣ If mask is not binary (e.g., values 0–1 or 0–255), threshold it
_, mask_binary = cv.threshold(mask_uint8, 127, 255, cv.THRESH_BINARY)

# 4️⃣ Apply bitwise_and
res = cv.bitwise_and(hsv_img, hsv_img, mask=mask_binary)

In [ ]:
final_image = mask_inv*hsv_img

In [ ]:
res = cv.bitwise_and(hsv_img, hsv_img, mask=head_masks)

In [ ]:
# mask = cv.inRange(hsv_img, head_masks)

ratio = cv.countNonZero(head_masks)/(hsv_img.size/3)
print('pixel percentage:', np.round(ratio*100, 2))
#plt.imshow(mask)

cv.imshow('mask',res)
cv.waitKey(1)

In [ ]:

cv.imshow('mask',res)
cv.waitKey(1)

In [ ]:
# Convert color for visualization (BGR → RGB or HSV → RGB)
res_rgb = cv2.cvtColor(res, cv2.COLOR_HSV2RGB)  # or COLOR_BGR2RGB if your image is in BGR

plt.figure(figsize=(6, 6))
plt.imshow(res_rgb)
plt.axis('off')
plt.title("Masked Image")
plt.show()

In [ ]:
res_rgb = cv2.cvtColor(result, cv2.COLOR_HSV2RGB)  # or COLOR_BGR2RGB if your image is in BGR

plt.figure(figsize=(6, 6))
plt.imshow(res_rgb)
plt.axis('off')
plt.title("Masked Image")
plt.show()